In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/nifty100.db")

ratios = pd.read_sql("SELECT * FROM financial_ratios", conn)

print(ratios.columns.tolist())

['id', 'company_id', 'year', 'net_profit_margin_pct', 'operating_profit_margin_pct', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'free_cash_flow_cr', 'capex_cr', 'earnings_per_share', 'book_value_per_share', 'dividend_payout_ratio_pct', 'total_debt_cr', 'cash_from_operations_cr', 'revenue_cagr_5yr', 'pat_cagr_5yr', 'eps_cagr_5yr', 'composite_quality_score']


In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/nifty100.db")

ratios = pd.read_sql("SELECT * FROM financial_ratios", conn)
market = pd.read_sql("SELECT * FROM market_cap", conn)
profit = pd.read_sql("SELECT * FROM profitandloss", conn)
sectors = pd.read_sql("SELECT * FROM sectors", conn)

ratios["financial_year"] = ratios["year"].str[:4]
market["financial_year"] = market["year"].astype(str)

latest = pd.merge(
    ratios,
    market,
    on=["company_id", "financial_year"],
    how="left",
)

latest.rename(
    columns={
        "year_x": "year",
        "year_y": "market_year",
    },
    inplace=True,
)

latest = pd.merge(
    latest,
    profit[["company_id", "year", "sales", "net_profit"]],
    on=["company_id", "year"],
    how="left",
)

latest = pd.merge(
    latest,
    sectors[["company_id", "broad_sector"]],
    on="company_id",
    how="left",
)

latest = latest[latest["year"] == "2024-03"]

print("PE <= 20 :", len(latest[latest["pe_ratio"] <= 20]))

print("PB <= 3 :", len(latest[latest["pb_ratio"] <= 3]))

print("Dividend Yield >= 1 :", len(latest[latest["dividend_yield_pct"] >= 1]))

PE <= 20 : 15
PB <= 3 : 10
Dividend Yield >= 1 : 74
